In [1]:
import warnings
warnings.filterwarnings("ignore")
import logging

import pandas as pd
import seaborn as sns
import numpy as np
import shap 
import torch
import os
import inspect 
import torch.nn as nn
import matplotlib.pyplot as plt
import pickle
from numpy import sqrt
from numpy import argmax
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve
from matplotlib import pyplot
from torchinfo import summary as torchsummary
from torchvision import models, transforms
from model import init_model
from matplotlib import pyplot
from pathlib import Path
sns.set(rc={'figure.figsize':(11.7,8.27)})
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999




# Parameters 

In [2]:
ROOT_DIR = Path(os.path.dirname(os.path.abspath('')))
TOTAL_TRAIN_SAMPLES = 50
TOTAL_TEST_SAMPLES = 10
CHANNELS = 3
DEVICE = "cuda:1"
OCT_PRESENCE = "Usando OCT"
DUAL_IMAGE = "Dual Image"
DATA_PATH = "../data.csv"
SUMMARY_PATH = "../model_summary.csv"
HISTORY_PATH = "../history.csv"
FT_SIZE = 24
ARQ1 = "SI"
ARQ2 = "SIT"
ARQ3 = "DI"
ARQ4 = "DIT"

# Functions

In [4]:
def get_samples_from_dataloader(dataloader, size):
    """
        Return photos1, photos2, ft_numerical, labels
    """ 
    
    list_of_photos_1 = []
    list_of_photos_2 = []
    list_of_features = []
    list_of_labels = []

    
    positive_class = 0
    negative_class = 0 
    
    for photos1, photos2, numericalft, labels in dataloader:
        for i in range(len(labels)):
            if int(labels[i].numpy()) == 0 and negative_class >= (size / 2):
                continue
            elif int(labels[i].numpy()) == 0 and negative_class < (size / 2):
                negative_class += 1
                
            if int(labels[i].numpy()) == 1 and positive_class >= (size / 2):
                continue 
            elif int(labels[i].numpy()) == 1 and positive_class < (size / 2):
                positive_class += 1
                
            list_of_photos_1.append(photos1[i])
            list_of_photos_2.append(photos2[i])
            list_of_features.append(numericalft[i])
            list_of_labels.append(labels[i].numpy())
        
        if len(list_of_labels) >= size:
            break
        
    
            
    photos1 = torch.stack(list_of_photos_1)
    photos2 = torch.stack(list_of_photos_2)
    ft_numerical = torch.stack(list_of_features)
    labels  = np.array(list_of_labels)
    
    return (photos1, photos2, ft_numerical, labels)


In [5]:
def load_model_and_loaders(model_row):
    """
        Return model, train_loader, val_loader
    """ 
    model_name = model_row["backbone"].values[0]
    pretrained = False
    feature_extract = False
    double_img_bool = True if model_row["double_img"].values[0] > 0 else False
    output_tab = int(model_row["output_tab"].values[0]) if model_row["output_tab"].values[0] > 0 else None 
    model, input_size = init_model(model_name,pretrained, feature_extract, double_img_bool, output_tab, FT_SIZE)
    model.load_state_dict(torch.load("../models/" + str(model_id) + "/" + "model"+ ".pth"))
    train_loader = torch.load("../models/" + str(model_id) + "/train_dataloader" + ".pth")
    val_loader = torch.load("../models/" + str(model_id) + "/val_dataloader" + ".pth")
    
    val_loader.dataset.root_dir = ROOT_DIR
    train_loader.dataset.root_dir = ROOT_DIR
    
    return model, train_loader, val_loader, input_size

In [8]:
def create_explainers(model, device, photos1_train, photos2_train, ft_numerical_train, 
                                     photos1_val, photos2_val, ft_numerical_val, input_size):
    """
        Return shap_values_photo1, shap_values_photo2, shap_values_nuermical
    """ 
    
    model = model.to(device)
    explainer = shap.DeepExplainer(model, [photos1_train.to(device),photos2_train.to(device),ft_numerical_train.to(device)])
    shap_values_photo1, shap_values_photo2, shap_values_numerical = explainer.shap_values([photos1_val.to(device),photos2_val.to(device), ft_numerical_val.to(device)])
    
    photos1_list = []
    photos2_list = []
    for i in range(len(photos1_val)):
        photos1_list.append(imshow(photos1_val[i]))
        photos2_list.append(imshow(photos2_val[i]))
        
    
    photos1_numpy = np.asarray(photos1_list)
    photos2_numpy = np.asarray(photos2_list)
    
    return (shap_values_photo1.reshape(-1, input_size, input_size, CHANNELS), 
            shap_values_photo2.reshape(-1, input_size, input_size, CHANNELS), 
            shap_values_numerical,
            photos1_numpy, 
            photos2_numpy)

In [9]:
def plot_image(shap_values, photos, total):
    
    return shap.image_plot(shap_values[:total], photos[:total])

In [10]:
def imshow(image):
    npimg = image.numpy()
    npimg = np.transpose(npimg, (1,2,0))
    npimg = ((npimg * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406])
    return npimg

In [11]:
def shapley_feature_ranking(shap_values):
    feature_order = np.argsort(np.mean(np.abs(shap_values), axis=0))
    return pd.DataFrame(
        {
            "features": [features_name[i] for i in feature_order][::-1],
            "importance": [
                np.mean(np.abs(shap_values), axis=0)[i] for i in feature_order
            ][::-1],
        }
    )

In [12]:
def get_rank_per_group(df_max,df_avg):
    dict_rows = {}
    dict_rows["Backbone"] = []
    dict_rows[OCT_PRESENCE] = []
    dict_rows[DUAL_IMAGE] = []
    dict_rows["Max Auc Rank"] = []
    dict_rows["Avg Auc Rank"] = []
 
 
    df_max = df_max.sort_values(by=["backbone",OCT_PRESENCE,DUAL_IMAGE])
    df_avg = df_avg.sort_values(by=["backbone",OCT_PRESENCE,DUAL_IMAGE])
    
    
    for index, row in df_max.iterrows():
        dict_rows["Backbone"].append(row["backbone"])
        dict_rows[OCT_PRESENCE].append(row[OCT_PRESENCE])
        dict_rows[DUAL_IMAGE].append(row[DUAL_IMAGE])
        dict_rows["Max Auc Rank"].append(row["rank_max_auc_oct_double"])
        
    for index, row in df_avg.iterrows():
        dict_rows["Avg Auc Rank"].append(row["rank_avg_auc_oct_double"])
        
    df_rank_per_group = pd.DataFrame.from_dict(dict_rows)
    return df_rank_per_group

# Preprocessing

In [13]:
data = pd.read_csv(DATA_PATH)
summary = pd.read_csv(SUMMARY_PATH)

In [16]:
summary.columns

Index(['model_id', 'k_fold', 'frac_val', 'val_best_acc', 'val_best_auc',
       'val_best_sp', 'val_best_sn', 'val_avg_acc', 'val_avg_auc',
       'val_avg_sp', 'val_avg_sn', 'train_best_acc', 'train_best_auc',
       'train_best_sp', 'train_best_sn', 'train_avg_acc', 'train_avg_auc',
       'train_avg_sp', 'train_avg_sn', 'host_name', 'optim', 'lr', 'scheduler',
       'randaugop', 'epochs', 'double_img', 'output_tab', 'backbone',
       'early_start', 'patient_el', 'timestamp', 'total_hours',
       'average_cross_hours', 'torchvision_version', 'torch_version',
       'history_added', 'shap_val', 'shap_oos', 'pred_val', 'pred_oos',
       'batch_size', 'Usando OCT', 'Dual Image', 'group_backbone',
       'regnet_vs_other', 'rank_max_auc', 'rank_avg_auc', 'Unnamed: 47',
       'Unnamed: 46', 'rank_max_auc_oct_double', 'rank_avg_auc_oct_double',
       'rank_avg', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 54'],
      dtype='object')

In [20]:
cross = summary.tail(28)

In [22]:
cross.sort_values("val_best_auc")

,model_id,k_fold,frac_val,val_best_acc,val_best_auc,val_best_sp,val_best_sn,val_avg_acc,val_avg_auc,val_avg_sp,val_avg_sn,train_best_acc,train_best_auc,train_best_sp,train_best_sn,train_avg_acc,train_avg_auc,train_avg_sp,train_avg_sn,host_name,optim,lr,scheduler,randaugop,epochs,double_img,output_tab,backbone,early_start,patient_el,timestamp,total_hours,average_cross_hours,torchvision_version,torch_version,history_added,shap_val,shap_oos,pred_val,pred_oos,batch_size,Usando OCT,Dual Image,group_backbone,regnet_vs_other,rank_max_auc,rank_avg_auc,Unnamed: 47,Unnamed: 46,rank_max_auc_oct_double,rank_avg_auc_oct_double,rank_avg,Unnamed: 50,Unnamed: 51,Unnamed: 54
862,a9607c10eea64a0487913d8b087664a5,5.0,NaN,0.868421,0.810740,1.000000,0.818182,0.776733,0.682587,0.899573,0.465601,0.886326,0.830382,0.997743,0.715976,0.811004,0.714366,0.935078,0.493653,magneto,sgd,0.0100,plateau,0,100.0,0.0,0.0,inception,30.0,10,2022-12-18 18:20:10.940602,25:24:52.66,05:04:58.50,0.13.0+cu113,1.12.0+cu113,1.0,0,0,0,0,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
860,de1ebc7a3d6441eab6255419c53cfd94,5.0,NaN,0.894737,0.858923,1.000000,0.941176,0.782866,0.712219,0.880475,0.543962,0.947195,0.931750,1.000000,0.897590,0.883244,0.833869,0.945991,0.721746,magneto,ranger,0.0005,plateau,0,100.0,0.0,0.0,inception,30.0,10,2022-12-17 23:59:53.626361,39:26:29.27,07:53:17.78,0.13.0+cu113,1.12.0+cu113,1.0,0,0,0,0,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
880,cc13a0cbc12f435d9438abf3eb8f71fe,5.0,NaN,0.888158,0.865155,1.000000,0.886364,0.811999,0.735675,0.911700,0.559650,0.958746,0.944165,1.000000,0.913793,0.896031,0.845923,0.960273,0.731573,magneto,radam,0.0010,plateau,0,100.0,0.0,5.0,regnet32x,30.0,10,2022-12-25 01:44:06.638228,41:32:33.44,08:18:30.67,0.13.0+cu113,1.12.0+cu113,1.0,0,0,0,0,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
861,5addd14041e54ea0b45050f8b2b840e2,5.0,NaN,0.888158,0.881854,1.000000,0.893617,0.809317,0.747094,0.892963,0.601225,0.940594,0.915429,0.981818,0.860335,0.897387,0.852200,0.955205,0.749195,magneto,radam,0.0010,plateau,0,100.0,0.0,0.0,regnet32x,30.0,10,2022-12-18 17:58:38.694663,25:03:17.29,05:00:39.44,0.13.0+cu113,1.12.0+cu113,1.0,0,0,0,0,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
859,1a500bc2ea8340bcbcad33a2f440b7b7,5.0,NaN,0.907895,0.888335,0.990654,0.970588,0.798468,0.723501,0.897845,0.549158,0.922442,0.888366,0.986301,0.932961,0.856809,0.794011,0.936859,0.651162,magneto,radam,0.0001,plateau,0,100.0,0.0,0.0,regnet16x,30.0,10,2022-12-17 23:08:14.658931,38:34:58.14,07:42:59.54,0.13.0+cu113,1.12.0+cu113,1.0,0,0,0,0,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
858,a50e719c914f4e07a6dc5fe36c191780,5.0,NaN,0.921053,0.898804,0.991150,0.957447,0.816615,0.758205,0.894366,0.622044,0.947195,0.931705,0.993151,0.893855,0.901829,0.858910,0.956714,0.761106,magneto,radam,0.0010,plateau,0,100.0,0.0,0.0,regnet16x,30.0,10,2022-12-17 23:05:14.946199,38:31:58.42,07:42:23.59,0.13.0+cu113,1.12.0+cu113,1.0,0,0,0,0,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
863,a948d6ee6c3f4fbb8804e4a51aea2890,5.0,NaN,0.921053,0.898804,1.000000,0.911765,0.791776,0.723413,0.886357,0.560469,0.945634,0.925988,1.000000,0.881657,0.883089,0.833037,0.946984,0.719090,magneto,ranger,0.0010,plateau,0,100.0,0.0,0.0,inception,30.0,10,2022-12-18 18:34:43.468383,25:39:25.19,05:07:53.01,0.13.0+cu113,1.12.0+cu113,1.0,0,0,0,0,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
869,564ce1d63d4643a5a14005274a6715a2,5.0,NaN,0.914474,0.903041,1.000000,0.911765,0.800148,0.722816,0.905366,0.540266,0.935644,0.917470,1.000000,0.877095,0.858810,0.789403,0.947451,0.631355,magneto,ranger,0.0010,plateau,0,100.0,0.0,5.0,shuffle,30.0,10,2022-12-20 18:38:01.790833,42:26:53.13,08:29:22.60,0.13.0+cu113,1.12.0+cu113,1.0,0,0,0,0,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
868,371f2592f8994c818e30a732c23aeaf6,5.0,NaN,0.921053,0.925972,1.000000,0.970588,0.822139,0.756661,0.911556,0.601765,0.966997,0.955490